In [5]:
import geopandas as gpd
wards_gdf = gpd.read_file('OSM_Chennai_155wards_2011.kml')
print(wards_gdf.columns)
print(wards_gdf.head())

Index(['id', 'Name', 'description', 'timestamp', 'begin', 'end',
       'altitudeMode', 'tessellate', 'extrude', 'visibility', 'drawOrder',
       'icon', 'ward', 'zone', 'geometry'],
      dtype='object')
     id  Name description timestamp begin end altitudeMode  tessellate  \
0  None  None        None       NaT   NaT NaT         None          -1   
1  None  None        None       NaT   NaT NaT         None          -1   
2  None  None        None       NaT   NaT NaT         None          -1   
3  None  None        None       NaT   NaT NaT         None          -1   
4  None  None        None       NaT   NaT NaT         None          -1   

   extrude  visibility  drawOrder  icon  ward         zone  \
0        0          -1        NaN  None   137   9-Saidapet   
1        0          -1        NaN  None   136   9-Saidapet   
2        0          -1        NaN  None   133   9-Saidapet   
3        0          -1        NaN  None   153  10-Mylapore   
4        0          -1        NaN  None

In [8]:
import pandas as pd
import numpy as np
import geopandas as gpd

# 0. Ward area from 2011 ward boundary file
wards_gdf = gpd.read_file('OSM_Chennai_155wards_2011.kml').copy()
wards_gdf['Ward'] = pd.to_numeric(wards_gdf['ward'], errors='coerce')

# project to a metric CRS before calculating area
wards_gdf = wards_gdf.to_crs(epsg=32644)
wards_gdf['Area_sqkm'] = wards_gdf.geometry.area / 10**6

area_df = wards_gdf[['Ward', 'Area_sqkm']].copy()


# 1 Population
pop_df = pd.read_excel('Census_Population_2011.xlsx', sheet_name= 'Sheet1')
pop_cols = ['Ward', 'Level', 'TOT_P', 'P_06', 'P_SC', 'P_ST', 'P_ILL']

# Filter Ward Level Data
pop_clean = pop_df[pop_df['Level'] == 'WARD'][pop_cols].copy()
pop_clean['Ward'] = pd.to_numeric(pop_clean['Ward'], errors='coerce')
pop_clean = pop_clean[pop_clean['Ward'] > 0]     # 0000 is district level
pop_clean['Ward'] = pop_clean['Ward'].astype(int)

pop_clean = pd.merge(pop_clean, area_df, on='Ward', how='inner')

# 1.1 Illiteracy Rate
# Literacy is measured for population aged 7 and above.
pop_clean['Olderthan7_Pop'] = pop_clean['TOT_P'] - pop_clean['P_06']
pop_clean['Illiteracy_Rate'] = np.where(
    pop_clean['Olderthan7_Pop'] > 0,
    pop_clean['P_ILL'] / pop_clean['Olderthan7_Pop'],
    np.nan
)

# 1.2 SC/ST Rate
pop_clean['SC_ST_Rate'] = np.where(
    pop_clean['TOT_P'] > 0,
    (pop_clean['P_SC'] + pop_clean['P_ST']) / pop_clean['TOT_P'],
    np.nan
)

# 1.3 Population Density
# Population/sqkm
pop_clean['Pop_Density'] = np.where(
    pop_clean['Area_sqkm'] > 0,
    pop_clean['TOT_P'] / pop_clean['Area_sqkm'],
    np.nan
)
# log transform
pop_clean['Pop_Density_Log'] = np.log1p(pop_clean['Pop_Density'])
# min-max normalization to 0-1
pd_min = pop_clean['Pop_Density_Log'].min()
pd_max = pop_clean['Pop_Density_Log'].max()
if pd_max > pd_min:
    pop_clean['Pop_Density_Norm'] = (
        (pop_clean['Pop_Density_Log'] - pd_min) / (pd_max - pd_min)
    )
else:
    pop_clean['Pop_Density_Norm'] = 0

# 2. Housing
housing_df = pd.read_excel('Census_Housing_2011.xlsx', sheet_name= 'Sheet1', header=None, skiprows=6)
housing_filtered = housing_df[housing_df[8].astype(str).str.contains('Ward No.', na=False)].copy()

housing_clean = pd.DataFrame()
housing_clean['Ward'] = pd.to_numeric(housing_filtered[7], errors='coerce')

housing_clean['Bad_Roof_Pct'] = (pd.to_numeric(housing_filtered[22], errors='coerce') + 
                                pd.to_numeric(housing_filtered[23], errors='coerce') + 
                                pd.to_numeric(housing_filtered[28], errors='coerce'))

housing_clean['Bad_Wall_Pct'] = (pd.to_numeric(housing_filtered[32], errors='coerce') +
                                pd.to_numeric(housing_filtered[33], errors='coerce') +
                                pd.to_numeric(housing_filtered[34], errors='coerce') +
                                pd.to_numeric(housing_filtered[35], errors='coerce') +
                                pd.to_numeric(housing_filtered[36], errors='coerce') +
                                pd.to_numeric(housing_filtered[38], errors='coerce'))

housing_clean['Bad_Floor_Pct'] = (pd.to_numeric(housing_filtered[42], errors='coerce') +
                                pd.to_numeric(housing_filtered[43], errors='coerce'))

housing_clean['Clean_Water_Pct'] = pd.to_numeric(housing_filtered[71], errors='coerce')

housing_clean['Insufficient_Drainage_Pct'] = (pd.to_numeric(housing_filtered[106], errors='coerce') + 
                                              pd.to_numeric(housing_filtered[107], errors='coerce'))

housing_clean['No_Assets_Pct'] = pd.to_numeric(housing_filtered[138], errors='coerce')

housing_clean = housing_clean[housing_clean['Ward'] > 0]        #0000 is district level


# 3. Final merge, ready for PCA
final_matrix = pd.merge(pop_clean[['Ward', 'Area_sqkm', 'Pop_Density', 'Pop_Density_Log', 'Pop_Density_Norm', 'SC_ST_Rate', 'Illiteracy_Rate']], 
                        housing_clean, on='Ward', how='inner')

final_matrix.to_csv('Chennai_SV_Clean.csv', index=False)

print("Feature Matrix Updated")
print(f"Effective Samples(Wards): {len(final_matrix)}")
print(final_matrix[['Ward', 'Area_sqkm', 'Illiteracy_Rate', 'Pop_Density', 'Bad_Floor_Pct']].head())

Feature Matrix Updated
Effective Samples(Wards): 155
   Ward  Area_sqkm  Illiteracy_Rate   Pop_Density  Bad_Floor_Pct
0     1   3.067874         0.195562  25020.584774            0.1
1     2   4.744462         0.208673  14100.019425            0.4
2     3   2.238140         0.275900  23678.147874            0.0
3     4   0.739842         0.264792  20525.997593            0.0
4     5   0.602173         0.256636  75068.112478            0.3
